In [35]:
import pandas as pd

In [36]:
df = pd.read_csv('../data/Raw_data/train.csv', index_col='id')
macro_df = pd.read_csv('../data/Raw_data/macro.csv')

In [41]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
macro_df['timestamp'] = pd.to_datetime(macro_df['timestamp'])
time_d = df.groupby(df['timestamp'].dt.to_period('M'))['price_doc'].count().reset_index()
time_d['N_count'] = time_d['price_doc'].cumsum()
time_d['d'] = time_d['N_count'] / sum(time_d['price_doc']) * 100
time_d

,timestamp,price_doc,N_count,d
0,2011-08,3,3,0.009845
1,2011-09,39,42,0.137836
2,2011-10,213,255,0.836861
3,2011-11,259,514,1.686850
4,2011-12,239,753,2.471202
5,2012-01,257,1010,3.314627
6,2012-02,364,1374,4.509205
7,2012-03,380,1754,5.756293
8,2012-04,300,2054,6.740836
9,2012-05,297,2351,7.715533


In [38]:
df["time_block"] = pd.qcut(
    df['timestamp'],
    q=6,
    labels=["B1", "B2", "B3", "B4", "B5", "B6"]
)

df_B = df.groupby("time_block").agg(
    n_transactions=("price_doc", "count"),
    min_date=("timestamp", "min"),
    max_date=("timestamp", "max"),
).reset_index()

df_B['time_diff'] = df_B['max_date'] - df_B['min_date']
df_B

C:\Users\sokol\AppData\Local\Temp\ipykernel_17180\470604544.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["time_block"] = pd.qcut(


,time_block,n_transactions,min_date,max_date,time_diff
0,B1,5101,2011-08-20,2012-12-04,472 days
1,B2,5064,2012-12-05,2013-08-29,267 days
2,B3,5072,2013-08-30,2014-02-19,173 days
3,B4,5109,2014-02-20,2014-06-28,128 days
4,B5,5099,2014-06-29,2014-11-27,151 days
5,B6,5026,2014-11-28,2015-06-30,214 days


In [42]:
df_train_full = df[df["time_block"].isin(["B1", "B2", "B3", "B4", "B5"])].copy().merge(macro_df, on='timestamp', how='left')
df_holdout_full = df[df["time_block"] == "B6"].copy().merge(macro_df, on='timestamp', how='left')
df_train_EDA = df[df["time_block"].isin(["B1", "B2", "B3", "B4", "B5"])].copy()
df_macro_EDA = df_train_full[macro_df.columns]


In [43]:
df_train_1 = df[df["time_block"].isin(["B1", "B2"])].copy().merge(macro_df, on='timestamp', how='left')
df_val_1 = df[df["time_block"].isin(["B3"])].copy().merge(macro_df, on='timestamp', how='left')

df_train_2 = df[df["time_block"].isin(["B1", "B2", "B3"])].copy().merge(macro_df, on='timestamp', how='left')
df_val_2 = df[df["time_block"].isin(["B4"])].copy().merge(macro_df, on='timestamp', how='left')

df_train_3 = df[df["time_block"].isin(["B1", "B2", "B3", "B4"])].copy().merge(macro_df, on='timestamp', how='left')
df_val_3 = df[df["time_block"].isin(["B5"])].copy().merge(macro_df, on='timestamp', how='left')

In [ ]:
df_train_1.to_csv("../data/train_test/val1/train1")
df_val_1.to_csv("../data/train_test/val1/val1")

df_train_2.to_csv("../data/train_test/val2/train2")
df_val_2.to_csv("../data/train_test/val2/val2")

df_train_3.to_csv("../data/train_test/val3/train3")
df_val_3.to_csv("../data/train_test/val3/val3")

In [45]:
df_train_full.to_csv("../data/train_test/holdout/train_full")
df_holdout_full.to_csv("../data/train_test/holdout/holdout")

df_train_EDA.to_csv("../data/EDA_data/train_EDA")
df_macro_EDA.to_csv("../data/EDA_data/macro_EDA")